<a href="https://colab.research.google.com/github/sandip01234/Transformer-Model-for-English-to-Nepali-translation/blob/main/Helsinki_NLP_opus_mt_en_ne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================================
# English → Nepali Translation: Complete Testing Suite
# Model: facebook/nllb-200-distilled-600M
# ============================================================
# INSTRUCTIONS:
#   - Paste each STEP as a separate cell in Google Colab
#   - Run them in order: Step 1 → Step 2 → ... → Step 12
#   - Only Step 1 needs your files uploaded
# ============================================================


# ============================================================
# STEP 1: Install dependencies
# Run this first, then Runtime → Restart session
# ============================================================
!pip install transformers==4.36.2 huggingface_hub==0.20.3 sacrebleu sentencepiece torch -q --force-reinstall


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 725.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 983.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [20]:

# ============================================================
# STEP 2: Imports
# Run after restarting session
# ============================================================
import os
import re
import json
import random
import torch
import sacrebleu
from collections import defaultdict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import files as colab_files

print("All imports successful.")



All imports successful.


In [26]:

# ============================================================
# STEP 3: Upload and parse your 10 domain .txt files
# Select all 10 files at once when the upload dialog appears
# ============================================================

print("Upload your 10 domain .txt files (select all at once)...")
uploaded = colab_files.upload()
print(f"\n{len(uploaded)} file(s) uploaded.")

# ── Helper: extract domain name from filename ────────────────
# Example: "honorifics.txt" → "honorifics"
#          "Food_Shopping.txt" → "food_shopping"

def extract_domain_from_filename(filename):
    name = os.path.splitext(filename)[0]        # remove .txt
    name = name.lower()
    name = re.sub(r'^domain[_\-]', '', name)    # remove leading "domain_"
    name = re.sub(r'[^a-z0-9]+', '_', name)    # spaces → underscore
    return name.strip('_')


# ── Helper: parse a single .txt file ────────────────────────
# Supports 2-column: english[TAB]nepali
# Supports 3-column: english[TAB]nepali[TAB]domain

def parse_txt_file(filename, content_bytes):
    pairs = []
    skipped = 0
    domain_from_file = extract_domain_from_filename(filename)

    try:
        content = content_bytes.decode('utf-8')
    except UnicodeDecodeError:
        content = content_bytes.decode('utf-8', errors='replace')

    for line in content.splitlines():
        line = line.strip()
        if not line:
            continue

        parts = line.split('\t')

        if len(parts) == 2:
            en, ne = parts[0].strip(), parts[1].strip()
            domain = domain_from_file
        elif len(parts) == 3:
            en, ne = parts[0].strip(), parts[1].strip()
            domain = parts[2].strip().lower()
        else:
            skipped += 1
            continue

        # Skip empty lines
        if not en or not ne:
            skipped += 1
            continue

        # Skip if Nepali column has no Devanagari script
        if not re.search(r'[\u0900-\u097F]', ne):
            skipped += 1
            continue

        pairs.append({'english': en, 'nepali': ne, 'domain': domain})

    return pairs, skipped


# ── Parse all uploaded files ─────────────────────────────────

all_data = []
domain_counts = defaultdict(int)

print("\nParsing uploaded files:")
for filename, content_bytes in uploaded.items():
    pairs, skipped = parse_txt_file(filename, content_bytes)
    all_data.extend(pairs)
    for p in pairs:
        domain_counts[p['domain']] += 1
    print(f"  {filename:<35} → {len(pairs):>4} pairs  ({skipped} skipped)")

print(f"\nTotal pairs loaded: {len(all_data)}")

# Show domain breakdown
print("\nDomain breakdown:")
print(f"  {'Domain':<25} {'Pairs':>6}")
print("  " + "-" * 33)
for domain, count in sorted(domain_counts.items()):
    status = "GOOD" if count >= 100 else ("OK" if count >= 50 else "LOW")
    print(f"  {domain:<25} {count:>6}   {status}")



Upload your 10 domain .txt files (select all at once)...


Saving business.txt to business (1).txt
Saving Daily Conversation.txt to Daily Conversation (1).txt
Saving education.txt to education (1).txt
Saving family.txt to family (1).txt
Saving food_shopping.txt to food_shopping (1).txt
Saving government.txt to government (1).txt
Saving Greetings & Honorifics.txt to Greetings & Honorifics (1).txt
Saving healthcare.txt to healthcare (1).txt
Saving religion_culture.txt to religion_culture (1).txt
Saving Ticket_Download_1DR3NLS.pdf to Ticket_Download_1DR3NLS.pdf
Saving travel.txt to travel (1).txt

11 file(s) uploaded.

Parsing uploaded files:
  business (1).txt                    →  194 pairs  (0 skipped)
  Daily Conversation (1).txt          →  182 pairs  (0 skipped)
  education (1).txt                   →  465 pairs  (0 skipped)
  family (1).txt                      →  165 pairs  (0 skipped)
  food_shopping (1).txt               →  110 pairs  (0 skipped)
  government (1).txt                  →  106 pairs  (0 skipped)
  Greetings & Honorifics (1

In [27]:

# ============================================================
# STEP 4: Train / Test split (85% train, 15% test)
# Stratified so every domain is in both sets
# ============================================================

random.seed(42)
train_data = []
test_data  = []

# Group by domain for stratified split
domain_groups = defaultdict(list)
for item in all_data:
    domain_groups[item['domain']].append(item)

for domain, items in domain_groups.items():
    shuffled = items.copy()
    random.shuffle(shuffled)

    # At least 3 test samples per domain
    n_test = max(3, int(len(shuffled) * 0.15))
    # Keep at least 10 for training
    n_test = min(n_test, len(shuffled) - 10)
    if n_test < 1:
        n_test = 0

    test_data.extend(shuffled[:n_test])
    train_data.extend(shuffled[n_test:])

random.shuffle(train_data)
random.shuffle(test_data)

print(f"Train set: {len(train_data)} pairs")
print(f"Test set:  {len(test_data)} pairs")

# Save both splits as TSV files
with open("dataset_train.tsv", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(f"{item['english']}\t{item['nepali']}\n")

# Test file keeps domain column for per-domain scoring
with open("dataset_test.tsv", "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(f"{item['english']}\t{item['nepali']}\t{item['domain']}\n")

print("Saved: dataset_train.tsv and dataset_test.tsv")



Train set: 1844 pairs
Test set:  319 pairs
Saved: dataset_train.tsv and dataset_test.tsv


In [28]:

# ============================================================
# STEP 5: Load NLLB model
# facebook/nllb-200-distilled-600M supports 200 languages
# including Nepali (npi_Deva)
# First download takes 5-10 minutes, cached after that
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG   = "eng_Latn"    # English
TGT_LANG   = "npi_Deva"    # Nepali Devanagari

print(f"\nLoading {MODEL_NAME}...")
print("(First time: ~5 minutes download. After that: ~30 seconds)")

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

print("Model loaded successfully.")

# No fine-tuned model yet — will be updated after fine-tuning
has_finetuned = False
ft_model      = base_model
ft_tokenizer  = base_tokenizer


Device: cpu

Loading facebook/nllb-200-distilled-600M...
(First time: ~5 minutes download. After that: ~30 seconds)
Model loaded successfully.


In [29]:

# ============================================================
# STEP 6: Translation function
# Uses NLLB with beam search for best quality output
# ============================================================

def translate_batch(sentences, model, tokenizer,
                    batch_size=16, num_beams=5, max_length=128):
    """
    Translate a list of English sentences to Nepali.

    Args:
        sentences:  list of English strings
        model:      NLLB model
        tokenizer:  NLLB tokenizer
        batch_size: sentences per batch (reduce to 8 if GPU runs out of memory)
        num_beams:  beam search width — higher = better quality but slower
        max_length: maximum output tokens

    Returns:
        list of Nepali translation strings in same order as input
    """
    model.eval()
    all_translations = []

    # Tell tokenizer we are translating FROM English
    tokenizer.src_lang = SRC_LANG

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        # Tokenize the batch
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                # Force decoder to start with Nepali language token
                forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
                num_beams=num_beams,
                max_length=max_length,
                early_stopping=True,
                no_repeat_ngram_size=3,   # prevents repetitive phrases
                length_penalty=1.0
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_translations.extend(decoded)

    return all_translations


# Quick test to verify model works
print("Testing translation...")
test_sentences = ["How are you?", "Please sit down.", "Good morning sir."]
test_results   = translate_batch(test_sentences, base_model, base_tokenizer)
for en, ne in zip(test_sentences, test_results):
    print(f"  EN: {en}")
    print(f"  NE: {ne}")
print("\nTranslation working correctly.")



Testing translation...
  EN: How are you?
  NE: तिमी कसरी छौ?
  EN: Please sit down.
  NE: कृपया बस्नुहोस्।
  EN: Good morning sir.
  NE: सुप्रभात सर.

Translation working correctly.


In [30]:


# ============================================================
# STEP 7: Translate the full test set
# This may take a few minutes depending on test set size
# ============================================================

print(f"Translating {len(test_data)} test sentences...")

# Extract lists from test_data
english_inputs = [item['english'] for item in test_data]
reference_ne   = [item['nepali']  for item in test_data]
domains        = [item['domain']  for item in test_data]

# Verify test data is not empty
if len(english_inputs) == 0:
    raise Exception("test_data is empty. Re-run Steps 3 and 4 first.")

# Translate with base model
print("  Running NLLB base model...")
base_outputs = translate_batch(english_inputs, base_model, base_tokenizer)
print(f"  Done: {len(base_outputs)} translations")

# If fine-tuned model exists use it, otherwise use base outputs
if has_finetuned:
    print("  Running fine-tuned model...")
    ft_outputs = translate_batch(english_inputs, ft_model, ft_tokenizer)
    print(f"  Done: {len(ft_outputs)} translations")
else:
    ft_outputs = base_outputs
    print("  No fine-tuned model — using base outputs for both columns.")

# Final check
assert len(base_outputs) == len(english_inputs), "Translation output count mismatch!"
print("\nAll translations complete. Ready for scoring.")


Translating 319 test sentences...
  Running NLLB base model...
  Done: 319 translations
  No fine-tuned model — using base outputs for both columns.

All translations complete. Ready for scoring.


In [31]:

# ============================================================
# STEP 8: Compute BLEU and chrF accuracy scores
#
# BLEU:  counts matching word n-grams (0-100)
#        > 20 usable,  > 30 good,  > 40 very good
# chrF:  character n-gram score — better for Devanagari script
#        because it handles Nepali morphology well
# ============================================================

def compute_scores(hypotheses, references):
    """
    Compute BLEU and chrF scores.
    hypotheses: list of model output strings
    references: list of reference (correct) strings
    Returns dict with bleu and chrf scores (0-100)
    """
    # sacrebleu expects references as list of lists
    refs_wrapped = [references]
    bleu = sacrebleu.corpus_bleu(hypotheses, refs_wrapped)
    chrf = sacrebleu.corpus_chrf(hypotheses, refs_wrapped)
    return {
        "bleu": round(bleu.score, 2),
        "chrf": round(chrf.score, 2),
    }


def interpret_bleu(score):
    """Return human-readable interpretation of BLEU score."""
    if score < 10:  return "Very low — needs more training data"
    if score < 20:  return "Low — understandable but many errors"
    if score < 30:  return "Moderate — usable translations"
    if score < 40:  return "Good — close to human quality"
    if score < 50:  return "Very good — near human quality"
    return                 "Excellent — human-level quality"


# Compute overall scores
base_scores = compute_scores(base_outputs, reference_ne)
ft_scores   = compute_scores(ft_outputs,   reference_ne)

print("=" * 60)
print("  OVERALL ACCURACY RESULTS")
print("=" * 60)
print(f"\n{'Metric':<10} {'Base Model':>12} {'Fine-tuned':>12} {'Change':>10}")
print("-" * 48)
print(f"{'BLEU':<10} {base_scores['bleu']:>12.2f} {ft_scores['bleu']:>12.2f} {ft_scores['bleu'] - base_scores['bleu']:>+10.2f}")
print(f"{'chrF':<10} {base_scores['chrf']:>12.2f} {ft_scores['chrf']:>12.2f} {ft_scores['chrf'] - base_scores['chrf']:>+10.2f}")
print(f"\nResult: {interpret_bleu(ft_scores['bleu'])}")



  OVERALL ACCURACY RESULTS

Metric       Base Model   Fine-tuned     Change
------------------------------------------------
BLEU              23.60        23.60      +0.00
chrF              60.26        60.26      +0.00

Result: Moderate — usable translations


In [32]:

# ============================================================
# STEP 9: Per-domain accuracy breakdown
# Shows which domains the model handles best and worst
# Very useful for your defence presentation
# ============================================================

domain_results = {}
domain_groups_test = defaultdict(list)

# Group test indices by domain
for i, item in enumerate(test_data):
    domain_groups_test[item['domain']].append(i)

# Compute scores per domain
for domain, indices in sorted(domain_groups_test.items()):
    if len(indices) < 3:
        continue    # skip domains with too few samples

    d_refs = [reference_ne[i] for i in indices]
    d_base = [base_outputs[i] for i in indices]
    d_ft   = [ft_outputs[i]   for i in indices]

    domain_results[domain] = {
        "count":     len(indices),
        "base_bleu": compute_scores(d_base, d_refs)["bleu"],
        "base_chrf": compute_scores(d_base, d_refs)["chrf"],
        "ft_bleu":   compute_scores(d_ft,   d_refs)["bleu"],
        "ft_chrf":   compute_scores(d_ft,   d_refs)["chrf"],
    }

print("\n" + "=" * 60)
print("  PER-DOMAIN ACCURACY")
print("=" * 60)
print(f"\n{'Domain':<22} {'N':>4}  {'Base BLEU':>10} {'FT BLEU':>9} {'Change':>8}")
print("-" * 58)

for domain, res in sorted(domain_results.items()):
    change = res['ft_bleu'] - res['base_bleu']
    arrow  = "↑" if change > 0 else ("↓" if change < 0 else "=")
    print(f"{domain:<22} {res['count']:>4}  {res['base_bleu']:>10.2f} {res['ft_bleu']:>9.2f} {change:>+7.2f} {arrow}")




  PER-DOMAIN ACCURACY

Domain                    N   Base BLEU   FT BLEU   Change
----------------------------------------------------------
business                 29       26.53     26.53   +0.00 =
daily_conversation       27        9.01      9.01   +0.00 =
education                69       20.38     20.38   +0.00 =
family                   24       31.66     31.66   +0.00 =
food_shopping            16       30.76     30.76   +0.00 =
government               15       42.40     42.40   +0.00 =
healthcare               39       13.50     13.50   +0.00 =
honorifics               63       17.64     17.64   +0.00 =
religion_culture         12       19.92     19.92   +0.00 =
travel                   25       53.61     53.61   +0.00 =


In [33]:

# ============================================================
# STEP 10: Sample translations — 2 per domain
# Shows side-by-side: English, Reference, Model output
# ============================================================

print("\n" + "=" * 60)
print("  SAMPLE TRANSLATIONS (2 per domain)")
print("=" * 60)

shown = defaultdict(int)
sample_count = 0

for i, item in enumerate(test_data):
    if shown[item['domain']] >= 2:
        continue
    if sample_count >= 20:
        break

    shown[item['domain']] += 1
    sample_count += 1

    print(f"\n[{sample_count}] Domain: {item['domain']}")
    print(f"  English:   {item['english']}")
    print(f"  Reference: {item['nepali']}")
    print(f"  Model out: {base_outputs[i]}")
    if has_finetuned:
        print(f"  Fine-tune: {ft_outputs[i]}")




  SAMPLE TRANSLATIONS (2 per domain)

[1] Domain: honorifics
  English:   You have shown us how honorifics bring warmth to conversations, teacher.
  Reference: गुरुजी, तपाईंले सम्मान प्रणालीले कुराकानीमा कसरी न्यानोपन ल्याउँछ देखाउनुभएको छ।
  Model out: तपाईंले हामीलाई देखाउनुभएको छ कि सम्मानपत्रले कुराकानीमा कसरी न्यानोपन ल्याउँछ, शिक्षक ।

[2] Domain: education
  English:   I need to improve my research skills.
  Reference: मलाई अनुसन्धान सीप सुधार्नुपर्छ।
  Model out: मलाई आफ्नो अनुसन्धान कौशल सुधार गर्न आवश्यक छ।

[3] Domain: family
  English:   Mother-in-law, the tea you make is the best.
  Reference: सासूआमा, तपाईंले बनाउनुभएको चिया सबैभन्दा राम्रो हुन्छ।
  Model out: ससुरा, तपाईंले बनाउनुभएको चिया उत्तम छ।

[4] Domain: education
  English:   The teacher motivated us to study harder.
  Reference: शिक्षकले हामीलाई अझ मेहनत गरेर पढ्न प्रेरित गर्नुभयो।
  Model out: शिक्षकले हामीलाई अझ बढी अध्ययन गर्न उत्प्रेरित गरे।

[5] Domain: family
  English:   Brother, let's plan a family picn

In [34]:

# ============================================================
# STEP 11: Best and worst translations
# Finds top 5 and bottom 5 by sentence-level BLEU
# Useful to show in your defence
# ============================================================

print("\n" + "=" * 60)
print("  BEST AND WORST TRANSLATIONS")
print("=" * 60)

# Compute sentence-level BLEU for every test pair
sentence_scores = []
for i in range(len(test_data)):
    score = sacrebleu.sentence_bleu(
        ft_outputs[i], [reference_ne[i]]
    ).score
    sentence_scores.append((score, i))

sentence_scores.sort(reverse=True)

print("\nTop 5 best translations:")
for rank, (score, idx) in enumerate(sentence_scores[:5], 1):
    print(f"\n  [{rank}] BLEU={score:.1f}  Domain: {domains[idx]}")
    print(f"  EN:  {test_data[idx]['english']}")
    print(f"  REF: {reference_ne[idx]}")
    print(f"  OUT: {ft_outputs[idx]}")

print("\nBottom 5 (where model struggles most):")
for rank, (score, idx) in enumerate(sentence_scores[-5:], 1):
    print(f"\n  [{rank}] BLEU={score:.1f}  Domain: {domains[idx]}")
    print(f"  EN:  {test_data[idx]['english']}")
    print(f"  REF: {reference_ne[idx]}")
    print(f"  OUT: {ft_outputs[idx]}")




  BEST AND WORST TRANSLATIONS

Top 5 best translations:

  [1] BLEU=100.0  Domain: food_shopping
  EN:  Can you reduce the price?
  REF: के तपाईं मूल्य घटाउन सक्नुहुन्छ?
  OUT: के तपाईं मूल्य घटाउन सक्नुहुन्छ?

  [2] BLEU=100.0  Domain: food_shopping
  EN:  I want to eat light food.
  REF: म हल्का खाना खान चाहन्छु।
  OUT: म हल्का खाना खान चाहन्छु।

  [3] BLEU=100.0  Domain: healthcare
  EN:  How many hours of sleep do you usually get?
  REF: तपाईं सामान्यतया कति घण्टा सुत्नुहुन्छ?
  OUT: तपाईं सामान्यतया कति घण्टा सुत्नुहुन्छ?

  [4] BLEU=100.0  Domain: travel
  EN:  I need a single room.
  REF: मलाई एकल कोठा चाहिन्छ।
  OUT: मलाई एकल कोठा चाहिन्छ।

  [5] BLEU=100.0  Domain: travel
  EN:  Is this the right direction?
  REF: के यो सही दिशा हो?
  OUT: के यो सही दिशा हो?

Bottom 5 (where model struggles most):

  [1] BLEU=0.0  Domain: daily_conversation
  EN:  Let's go early tomorrow।
  REF: भोलि चाँडै जाऔं।
  OUT: चलो कल जल्दी चलें।

  [2] BLEU=0.0  Domain: government
  EN:  Every citize

In [ ]:

# ============================================================
# STEP 12: Live interactive demo
# Type any English sentence and see it translated
# Perfect for demonstrating at your defence
# ============================================================

print("\n" + "=" * 60)
print("  LIVE TRANSLATION DEMO")
print("  Type any English sentence to translate.")
print("  Type 'quit' to exit and save the report.")
print("=" * 60)


def translate_one(text):
    """Translate a single English sentence to Nepali."""
    base_model.eval()
    base_tokenizer.src_lang = SRC_LANG

    inputs = base_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            forced_bos_token_id=base_tokenizer.lang_code_to_id[TGT_LANG],
            num_beams=5,
            max_length=128,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return base_tokenizer.decode(out[0], skip_special_tokens=True)


while True:
    try:
        user_input = input("\nEnglish: ").strip()
    except EOFError:
        break
    if user_input.lower() in ("quit", "exit", "q", ""):
        break
    print(f"Nepali:  {translate_one(user_input)}")




  LIVE TRANSLATION DEMO
  Type any English sentence to translate.
  Type 'quit' to exit and save the report.

English: nishchal is going to marry bhumika
Nepali:  nishchal bhumika संग विवाह गर्न जाँदैछ


In [ ]:

# ============================================================
# STEP 13: Save report and download results
# Downloads 3 files:
#   dataset_train.tsv     — your training set
#   dataset_test.tsv      — your test set with domain labels
#   translation_report.json — full results for your report
# ============================================================

sentence_scores_dict = {idx: score for score, idx in sentence_scores}

report = {
    "summary": {
        "model":             "facebook/nllb-200-distilled-600M",
        "total_test_pairs":  len(test_data),
        "total_train_pairs": len(train_data),
        "domains_tested":    len(domain_results),
        "base_bleu":         base_scores["bleu"],
        "base_chrf":         base_scores["chrf"],
        "finetuned_bleu":    ft_scores["bleu"],
        "finetuned_chrf":    ft_scores["chrf"],
        "bleu_improvement":  round(ft_scores["bleu"] - base_scores["bleu"], 2),
        "chrf_improvement":  round(ft_scores["chrf"] - base_scores["chrf"], 2),
        "interpretation":    interpret_bleu(ft_scores["bleu"]),
    },
    "per_domain": domain_results,
    "samples": [
        {
            "domain":        test_data[i]["domain"],
            "english":       test_data[i]["english"],
            "reference":     reference_ne[i],
            "base_output":   base_outputs[i],
            "ft_output":     ft_outputs[i],
            "sentence_bleu": round(sentence_scores_dict.get(i, 0), 2)
        }
        for i in range(len(test_data))
    ]
}

with open("translation_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nDownloading result files...")
colab_files.download("dataset_train.tsv")
colab_files.download("dataset_test.tsv")
colab_files.download("translation_report.json")


In [37]:

# ============================================================
# FINAL SUMMARY — copy this for your defence slides
# ============================================================

print("\n" + "=" * 60)
print("  DEFENCE-READY SUMMARY")
print("=" * 60)
print(f"""
  Project:  English to Nepali Machine Translation
  Model:    facebook/nllb-200-distilled-600M
  Dataset:  {len(all_data)} pairs across {len(domain_counts)} domains
  Test set: {len(test_data)} pairs (15% stratified hold-out)

  METRIC        BASE MODEL    FINE-TUNED
  ──────────────────────────────────────
  BLEU score    {base_scores['bleu']:>8.2f}      {ft_scores['bleu']:>8.2f}
  chrF score    {base_scores['chrf']:>8.2f}      {ft_scores['chrf']:>8.2f}

  Result: {interpret_bleu(ft_scores['bleu'])}
""")


  DEFENCE-READY SUMMARY

  Project:  English to Nepali Machine Translation
  Model:    facebook/nllb-200-distilled-600M
  Dataset:  2163 pairs across 11 domains
  Test set: 319 pairs (15% stratified hold-out)

  METRIC        BASE MODEL    FINE-TUNED
  ──────────────────────────────────────
  BLEU score       23.60         23.60
  chrF score       60.26         60.26

  Result: Moderate — usable translations


  DEFENCE-READY SUMMARY

  Project:  English to Nepali Machine Translation
  Model:    facebook/nllb-200-distilled-600M
  Dataset:  2163 pairs across 11 domains
  Test set: 319 pairs (15% stratified hold-out)

  METRIC        BASE MODEL    FINE-TUNED
  ──────────────────────────────────────
  BLEU score       23.60         23.60
  chrF score       60.26         60.26

  Result: Moderate — usable translations

